# 0. Imports

In [82]:
import numpy as numpy
import torch
import torch.nn as nn
import torch.optim as optim 
from torch.distributions import Categorical
import gymnasium as gym
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm
import time
import math

In [83]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# 1. Hyperparamters

In [ ]:
hyperparameters = {
    "actor" : {
        "optimizer" : "adam",
        "alpha" : 0.001,
        "dropout" : 0,
        "negative_slope" : 0.01,
        "verbose" : False
    },
    "critic" : {
        "optimizer" : "adam",
        "alpha" : 0.001,
        "dropout" : 0,
        "negative_slope" : 0.01,
        "verbose" : False
    }, 
    "environment" : {
        "gamma" : 0.9,
        "episodes" : 100000,
        "name" : "CartPole-v1",
        "render_mode": None
    },
    "use_tensorboard" : True,
    "tb_run_dir_path" : "runs/ActorCritic/CartPole-v1/"

}

In [85]:
expected_schema = {
    "actor": {
        "optimizer": str,
        "alpha": float,
        "dropout": float,
        "negative_slope": float,
        "verbose": bool
    },
    "critic": {
        "optimizer": str,
        "alpha": float,
        "dropout": float,
        "negative_slope": float,
        "verbose": bool
    },
    "environment": {
        "gamma": float,
        "episodes": int,
        "name": str,
        "render_mode": (str, type(None)),
    },
    "use_tensorboard" : bool,
    "tb_run_dir_path" : str
}

def validate_types(d: dict, schema: dict, path=""):
    errors = []

    for key in schema:
        full_key = f"{path}.{key}" if path else key
        if key not in d:
            errors.append(f"Missing key: {full_key}")
            continue

        expected_type = schema[key]
        actual_value = d[key]

        if isinstance(expected_type, dict):
            if not isinstance(actual_value, dict):
                errors.append(f"Type mismatch at {full_key}: expected dict, got {type(actual_value).__name__}")
            else:
                errors.extend(validate_types(actual_value, expected_type, full_key))
        else:
            if not isinstance(actual_value, expected_type):
                errors.append(
                    f"Type mismatch at {full_key}: expected {expected_type.__name__}, got {type(actual_value).__name__}"
                )

    return errors

# 2. Actor NN

In [86]:
class Actor(nn.Module):
    """
    Class (nn.Module): Simple NN to act as policy learner
    """
    def __init__(self, state_dim: int, action_dim:int, dropout: float = 0.2, negative_slope: float = 0):
        """
        Function: Initialises the nn.Module obj to create a policy learner a.k.a. Actor
        Args:
            state_dim (int): Number of state dimension to in env to consider as obervable
            action_dim (int): Number of action dimensions to output the action probablities
            dropout (0 <= float <= 1): The dropout value for the final dropout layer
            negative_slope (0 <= int <=1): The slope angle of negtive units if we want to control for LeakyReLU()
        Returns: None
        """
        super(Actor, self).__init__()

        # Init NN with 
        self.model = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.Dropout(p=dropout),
            nn.LeakyReLU(negative_slope=negative_slope),
            nn.Linear(128, action_dim),
            nn.Dropout(p=dropout),
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        """
        Function: Forward loop of the policy learner
        Args: 
            state (tensor): It contains the following features
                - x : Position of the cart from the center
                - x_dot : Velocity of the cart horizontally
                - theeta : Pole Angle from the vertical
                - theeta_dot : Angular velocity of the pole

        Return: 
            action (tensor): Outputs a vector with the probablities of the actions that can be taken
        """

        return self.model(state)     

# 3. Critic Model

In [87]:
class Critic(nn.Module):
    """
    Class (nn.Module): Simple NN to act as policy learner
    """

    def __init__(self, state_dim:int , dropout: float = 0.2, negative_slope: float = 0):
        """
        Function: Initialises the nn.Module obj to create a value function estimator a.k.a. Critic
        Args:
            state_dim (int): Number of state dimension to in env to consider as obervable
            dropout (0 <= float <= 1): The dropout value for the final dropout layer
            negative_slope (0 <= int <=1): The slope angle of negtive units if we want to control for LeakyReLU()
        Returns: None
        """

        super(Critic, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.LeakyReLU(negative_slope=negative_slope),
            nn.Dropout(p=dropout),
            nn.Linear(128,1)
        )

    def forward(self, state):
        """
        Function: Returns expected total future reward from state s
        Args: 
            state (tensor): It contains the following features
                - x : Position of the cart from the center
                - x_dot : Velocity of the cart horizontally
                - theeta : Pole Angle from the vertical
                - theeta_dot : Angular velocity of the pole
        Returns: Value estimate of the given state
        """
        return self.model(state)

# 4. Training Loop for Actor-Critic

In [88]:
def train_actor_critic(hyperparameters: dict):
    """
    Function: Trains the actor and critic function approximators simultaneous to achieve optimality on the CartPole=v1 example
    Args:
        hyperparameters
    """

    # Validate Hyper Parameter inputs input 
    errors = validate_types(hyperparameters, expected_schema)

    # If errors found return None
    if errors:
        print("Type validation errors found:")
        for error in errors:
            print(" -", error)

        return None, None, None
    
    # Initialise the environment
    # Get the state and action dimension
    env = gym.make(id=hyperparameters['environment']['name'], render_mode=hyperparameters['environment']['render_mode'])
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    # Load the actor and critic function approximators
    actor = Actor(state_dim=state_dim, 
                  action_dim=action_dim, 
                  dropout=hyperparameters['actor']['dropout'],
                  negative_slope=hyperparameters['actor']['negative_slope']).to(device)
    
    critic = Critic(state_dim=state_dim,
                    dropout=hyperparameters['critic']['dropout'],
                    negative_slope=hyperparameters['critic']['negative_slope']).to(device)
    
    # Initialise the Optimizers for the 
    actor_optimizer = optim.Adam(params=actor.parameters(), lr= hyperparameters['actor']['alpha'])
    critic_optimizer = optim.Adam(params=critic.parameters(), lr=hyperparameters['critic']['alpha'])

    rewards_history = []

    # Initialise a SummaryWriter
    writer = SummaryWriter(log_dir=f"{hyperparameters['tb_run_dir_path']}_run_{math.floor(time.time())}") if hyperparameters['use_tensorboard'] else None

    # For each episode
    for episode in tqdm(range(hyperparameters['environment']['episodes']), desc="Training Actor-Critic Models Simultaneously: "):

        # Reset the env
        # define the completion flag 
        state, _ = env.reset() 
        done = False
        total_reward = 0

        # Execute step by step trajectory
        while not done:

            # Typecast and move state tensor to device
            state_tensor = torch.tensor(state, dtype=torch.float32).to(device)

            # Get the action probablities from the neural net 
            # Define the actionprobablities as a categorical 
            # Sample the action probablities and choose an action based on the dist
            action_probs = actor(state_tensor)
            action_dist = torch.distributions.Categorical(action_probs)
            action = action_dist.sample()
            
            # Take the action in the environment
            next_state, reward, terminated, truncated, _ = env.step(action=action.item())
            done = terminated or truncated
            total_reward += reward

            # Typecast and move the reward tenors to GPU
            next_state_tensor = torch.tensor(next_state, dtype=torch.float32).to(device)
            reward_tensor = torch.tensor(reward, dtype=torch.float32).to(device)

            value_curr = critic(state_tensor)
            value_next = critic(next_state_tensor).detach()
            td_target = reward_tensor + hyperparameters['environment']['gamma'] * value_next * (1 - int(done))
            td_error = td_target - value_curr                

            # Update the Critic 
            critic_loss = td_error.pow(2).mean()
            critic_optimizer.zero_grad()
            critic_loss.backward()
            critic_optimizer.step()

            # Update the Actor
            log_prob = action_dist.log_prob(action)
            actor_loss = -log_prob * td_error.detach()
            actor_loss.backward()
            actor_optimizer.step()

            state = next_state

        # Update the rewrd history
        rewards_history.append(total_reward)

        # Log at tensor board
        if hyperparameters['use_tensorboard']:
            writer.add_scalar("Reward/Episode", total_reward, episode)
            writer.add_scalar("Loss/Actor", actor_loss.item(), episode)
            writer.add_scalar("Loss/Critic", critic_loss.item(), episode)

    # Close writer 
    # Close env
    # Return actor, critic and the rewards_history
    writer.close()
    env.close()
    return actor, critic, rewards_history


In [89]:
actor, critic, rewards_history = train_actor_critic(hyperparameters=hyperparameters)

Training Actor-Critic Models Simultaneously:   0%|          | 0/1000 [00:00<?, ?it/s]